In [ ]:
import importlib.util
import subprocess
import sys

required = {
    'openai': 'openai', 'pandas': 'pandas', 'numpy': 'numpy',
    'scipy': 'scipy', 'sklearn': 'scikit-learn', 'PIL': 'pillow',
    'tqdm': 'tqdm', 'nbformat': 'nbformat',
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import OrderedDict
import base64
import hashlib
import json
import math
import os
import re
import time

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image
from scipy.special import expit, logit
from scipy.stats import rankdata, spearmanr
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from openai import OpenAI

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    drive = userdata = None
    IN_COLAB = False

RANDOM_SEED = 20260804
np.random.seed(RANDOM_SEED)

                                                   
RUN_TRAIN_API_CALLS = False
RUN_DEV_API_CALLS = False
RUN_TEST_API_CALLS = False
RUN_LIMIT = None

CI_MODEL = 'gpt-5.5'
CI_REASONING_EFFORT = 'none'
CI_IMAGE_DETAIL = 'original'
REQUEST_SLEEP_SECONDS = 0.20
MAX_RETRIES = 3

OUTER_FOLDS = 5
INNER_FOLDS = 4
RIDGE_ALPHAS = [0.1, 1.0, 10.0, 100.0]
BLEND_WEIGHTS = np.round(np.linspace(0.0, 1.0, 11), 2).tolist()

                                                           
PRE_API_MIN_VERSION_PRIOR_RHO = 0.70
PRE_API_MAX_VERSION_PRIOR_MAE = 0.19
TRAIN_GATE_MIN_RHO = 0.75
TRAIN_GATE_MIN_DELTA_RHO = 0.03
TRAIN_GATE_MAX_MAE = 0.19
TRAIN_GATE_MIN_FOLD_WINS = 4

print('Colab:', IN_COLAB)
print('Model:', CI_MODEL)
print('API switches:', RUN_TRAIN_API_CALLS, RUN_DEV_API_CALLS, RUN_TEST_API_CALLS)


In [ ]:
if IN_COLAB:
    drive.mount('/content/drive')

default_root = Path('/content/drive/MyDrive/Dr. Lulwah - Ahmed/ImageEVAl')
PROJECT_DIR = Path(os.environ.get(
    'IMAGEEVAL_PROJECT_DIR',
    str(default_root / 'ImageEval2026_Task2_CRAI_Bench'),
))

data_override = os.environ.get('IMAGEEVAL_DATA_DIR')
if data_override:
    DATA_DIR = Path(data_override)
else:
    candidates = [PROJECT_DIR / 'data', default_root / 'train_dev']
    DATA_DIR = next(
        (p for p in candidates if (p / 'train' / 'captions.tsv').exists()),
        candidates[0],
    )

EXPERIMENT_ROOT = PROJECT_DIR / 'ci_integrity_structured_v1'
CACHE_DIR = EXPERIMENT_ROOT / 'cache'
OUTPUT_DIR = EXPERIMENT_ROOT / 'outputs'
for path in [CACHE_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_DIR)
print('Data:', DATA_DIR)
print('Experiment:', EXPERIMENT_ROOT)


In [ ]:
DIM_COLS = ['CRAI_CEA', 'CRAI_CC', 'CRAI_CS', 'CRAI_CI', 'CRAI_HP']

def find_image(folder: Path, stem: str) -> Path:
    for ext in ['.png', '.jpg', '.jpeg', '.webp']:
        candidate = folder / f'{stem}{ext}'
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'No image found for {stem!r} in {folder}')

def load_split(split: str, require_gold: bool) -> pd.DataFrame:
    split_dir = DATA_DIR / split
    captions = pd.read_csv(split_dir / 'captions.tsv', sep='\t')
    if require_gold:
        gold = pd.read_csv(split_dir / 'gold_human.tsv', sep='\t')
        keep = ['id'] + [c for c in DIM_COLS + ['CRAI_composite', 'category'] if c in gold.columns]
        frame = captions.merge(gold[keep], on='id', how='inner', validate='one_to_one')
    else:
        frame = captions.copy()

    frame['id'] = frame['id'].astype(str)
    frame['base_id'] = frame['id'].str.replace(r'_v\d+$', '', regex=True)
    frame['caption_version'] = frame['id'].str.extract(r'_v(\d+)$')[0].astype(int)
    frame['caption_version_key'] = 'v' + frame['caption_version'].astype(str)
    if 'image_id' in frame.columns:
        assert frame['base_id'].eq(frame['image_id'].astype(str)).all()
    return frame.sort_values(['base_id', 'caption_version']).reset_index(drop=True)

train_df = load_split('train', require_gold=True)
dev_df = load_split('dev', require_gold=True)
test_path = DATA_DIR / 'test' / 'captions.tsv'
test_df = load_split('test', require_gold=False) if test_path.exists() else pd.DataFrame()

assert len(train_df) == 120 and train_df['base_id'].nunique() == 24
assert len(dev_df) == 40 and dev_df['base_id'].nunique() == 8
assert train_df.groupby('base_id')['caption_version'].nunique().eq(5).all()
assert dev_df.groupby('base_id')['caption_version'].nunique().eq(5).all()

print('train:', train_df.shape, '| groups:', train_df['base_id'].nunique())
print('dev:', dev_df.shape, '| groups:', dev_df['base_id'].nunique())
print('test:', test_df.shape if len(test_df) else 'not released / not found')
display(train_df[['id', 'caption_version_key', 'CRAI_CI', 'caption']].head())


In [ ]:
ci_summary = train_df['CRAI_CI'].describe().to_frame('value')
ci_by_version = train_df.groupby('caption_version_key')['CRAI_CI'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
)
ci_correlations = train_df[DIM_COLS].corr(method='spearman')

display(ci_summary.round(4))
display(ci_by_version.round(4))
display(ci_correlations.round(4))

print('CI score frequencies:')
display(train_df['CRAI_CI'].value_counts().sort_index().rename_axis('score').to_frame('count'))


In [ ]:
PROMPT_VERSION = 'ci-cultural-integrity-structured-v1'
SCHEMA_VERSION = 'ci-cultural-integrity-schema-v1'

CI_SYSTEM_PROMPT = r'''
ROLE

You are a strict multimodal evaluator for ImageEval 2026 CRAI-Bench. Judge only
CRAI_CI: Cultural Integrity for the Qatari cultural scene represented by the inputs.

AUTHOR DEFINITION

Cultural Integrity is the respectful and undistorted preservation of cultural
meaning.

INPUT AUTHORITY

- The REFERENCE IMAGE represents the intended cultural subject, practice, object,
  garment, place, or landmark.
- The V1 CAPTION names and explains culturally explicit identity-bearing details.
- The CURRENT CAPTION is the prompt used to generate the candidate. It may be more
  generic, but its loss of detail does not erase cultural meaning established by the
  reference and V1 caption.
- The GENERATED IMAGE is the candidate being scored.

DECISION PROCEDURE

1. Identify the smallest set of core features or relationships that carry the
   intended cultural meaning in the reference and V1 caption.
2. Decide whether the generated image still depicts that cultural subject or
   practice, rather than only satisfying a generic paraphrase.
3. Judge whether the identity-bearing form, use, handling, clothing construction,
   landmark geometry, or practice has been materially distorted.
4. Judge whether the representation is culturally plausible and respectful.
5. Assign the holistic CI score from the anchors below. Do not mechanically average
   the diagnostic fields.

SCORE ANCHORS

- 1.00: The intended cultural meaning is clearly recognizable, plausible,
  respectful, and materially undistorted. Composition and incidental detail may differ.
- 0.75: The core meaning is preserved. Noticeable changes or minor genericization
  exist, but identity and cultural function remain intact.
- 0.50: Cultural meaning is only partly preserved. A generic substitute, altered
  identity-bearing form, or important distortion weakens the intended meaning, but a
  meaningful portion remains recognizable.
- 0.25: Only a weak fragment or broad theme survives. Major genericization or
  distortion has removed most of the intended cultural meaning.
- 0.00: The intended cultural subject or practice is absent, replaced,
  unrecognizable, or materially corrupted, even if the generic current caption is met.

BOUNDARIES WITH OTHER CRAI METRICS

- Do not score CEA. Missing objects, counts, colors, or attributes affect CI only
  when their loss materially changes cultural identity or meaning.
- Do not score CC. Spatial or relational errors affect CI only when they corrupt the
  meaning of the practice, object, or landmark.
- Do not score CS. A scene may be less Qatar-specific yet retain integrity if its
  intended cultural meaning remains recognizable and undistorted.
- Do not score HP. Unsupported additions matter only when they distort meaning or
  make the representation disrespectful.
- Do not score general beauty, realism, lighting, or photographic quality unless a
  defect materially damages the cultural representation.

CRITICAL RULES

- A respectful generic scene is not enough when the intended identity is lost.
- Superficial resemblance or shared function is not enough for a named landmark,
  practice, garment, or cultural object.
- Do not require pixel-level similarity, identical camera angle, or every reference
  detail.
- Do not assume that traditional-looking, Arab-looking, Gulf-looking, or desert
  imagery automatically preserves the intended Qatari cultural meaning.
- Keep brief_reason to one sentence and return JSON only.

OUTPUT JSON

{
  "core_identity_retention": 0.0,
  "practice_or_form_integrity": 0.0,
  "respect_and_plausibility": 0.0,
  "generic_substitution_severity": 0.0,
  "material_distortion_severity": 0.0,
  "integrity_failure": "none | minor | substantial | total",
  "raw_ci": 0.0,
  "confidence": 0.0,
  "brief_reason": "one short sentence"
}
'''.strip()

def short_hash(value: str, length: int = 10) -> str:
    return hashlib.sha256(value.strip().encode('utf-8')).hexdigest()[:length]

def cache_tag() -> str:
    return (
        f'{PROMPT_VERSION}_{CI_MODEL}_reasoning-{CI_REASONING_EFFORT}_'
        f'detail-{CI_IMAGE_DETAIL}_prompt-{short_hash(CI_SYSTEM_PROMPT)}_'
        f'schema-{short_hash(SCHEMA_VERSION)}_demos-none'
    )

def cache_path(split: str) -> Path:
    return CACHE_DIR / f'ci_structured_{split}_{cache_tag()}.jsonl'

def attempt_path(split: str) -> Path:
    return CACHE_DIR / f'ci_structured_attempts_{split}_{cache_tag()}.jsonl'

print('Prompt hash:', short_hash(CI_SYSTEM_PROMPT))
print('Cache family:', cache_tag())


In [ ]:
FAILURE_LEVELS = ['none', 'minor', 'substantial', 'total']
REQUIRED_NUMERIC = [
    'core_identity_retention', 'practice_or_form_integrity',
    'respect_and_plausibility', 'generic_substitution_severity',
    'material_distortion_severity', 'raw_ci', 'confidence',
]

def unit_float(value, name: str) -> float:
    value = float(value)
    if not np.isfinite(value) or not 0.0 <= value <= 1.0:
        raise ValueError(f'{name} must be a finite number in [0, 1], got {value!r}')
    return value

def parse_json_object(text: str) -> dict:
    text = text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
    start, end = text.find('{'), text.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('No JSON object found')
    return json.loads(text[start:end + 1])

def validate_response(value: dict) -> dict:
    required = set(REQUIRED_NUMERIC + ['integrity_failure', 'brief_reason'])
    missing = required - set(value)
    if missing:
        raise ValueError(f'Missing response fields: {sorted(missing)}')
    failure = str(value['integrity_failure']).strip().lower()
    if failure not in FAILURE_LEVELS:
        raise ValueError(f'Invalid integrity_failure: {failure!r}')
    reason = str(value['brief_reason']).strip()
    if not reason:
        raise ValueError('brief_reason is empty')
    result = {name: unit_float(value[name], name) for name in REQUIRED_NUMERIC}
    result['integrity_failure'] = failure
    result['brief_reason'] = reason
    return result

def load_jsonl(path: Path) -> list:
    if not path.exists():
        return []
    records = []
    with path.open('r', encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except Exception as exc:
                    raise ValueError(f'Invalid JSONL at {path}:{line_number}') from exc
    return records

def append_jsonl(path: Path, record: dict):
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
        handle.flush()

def image_to_data_url(path: Path) -> str:
    mime = {'.png': 'image/png', '.jpg': 'image/jpeg', '.jpeg': 'image/jpeg', '.webp': 'image/webp'}[path.suffix.lower()]
    data = base64.b64encode(path.read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{data}'

def split_image_paths(split: str, row: pd.Series):
    image_root = DATA_DIR / split / 'imgs'
    ref = find_image(image_root / 'ref', str(row['base_id']))
    generated = find_image(image_root / 'generated', str(row['id']))
    return ref, generated

def v1_caption_map(frame: pd.DataFrame) -> dict:
    v1 = frame.loc[frame['caption_version'].eq(1), ['base_id', 'caption']]
    if v1['base_id'].duplicated().any():
        raise ValueError('Duplicate v1 captions')
    return dict(zip(v1['base_id'].astype(str), v1['caption'].astype(str)))

client = None
def get_client():
    global client
    if client is None:
        if not IN_COLAB:
            key = os.environ.get('OPENAI_API_KEY')
        else:
            key = userdata.get('openai')
        if not key:
            raise RuntimeError('OpenAI API key not found. In Colab, create secret "openai".')
        client = OpenAI(api_key=key)
    return client

def call_ci_judge(row: pd.Series, v1_caption: str, split: str) -> dict:
    ref_path, gen_path = split_image_paths(split, row)
    user_text = f'''Instance ID: {row['id']}

V1 culturally explicit caption:
{v1_caption}

Current caption used to generate the candidate:
{row['caption']}

The first image is the REFERENCE IMAGE. The second image is the GENERATED IMAGE.
Judge only Cultural Integrity and return the required JSON.'''.strip()

    for attempt in range(1, MAX_RETRIES + 1):
        timestamp = datetime.now(timezone.utc).isoformat()
        try:
            response = get_client().responses.create(
                model=CI_MODEL,
                reasoning={'effort': CI_REASONING_EFFORT},
                input=[
                    {'role': 'system', 'content': [{'type': 'input_text', 'text': CI_SYSTEM_PROMPT}]},
                    {'role': 'user', 'content': [
                        {'type': 'input_text', 'text': user_text},
                        {'type': 'input_text', 'text': 'REFERENCE IMAGE:'},
                        {'type': 'input_image', 'image_url': image_to_data_url(ref_path), 'detail': CI_IMAGE_DETAIL},
                        {'type': 'input_text', 'text': 'GENERATED IMAGE:'},
                        {'type': 'input_image', 'image_url': image_to_data_url(gen_path), 'detail': CI_IMAGE_DETAIL},
                    ]},
                ],
            )
            parsed = validate_response(parse_json_object(response.output_text))
            record = {
                'instance_id': str(row['id']), 'split': split,
                'prompt_version': PROMPT_VERSION, 'prompt_hash': short_hash(CI_SYSTEM_PROMPT),
                'schema_version': SCHEMA_VERSION, 'model': CI_MODEL,
                'reasoning_effort': CI_REASONING_EFFORT, 'image_detail': CI_IMAGE_DETAIL,
                'demonstrations': [], 'created_utc': timestamp, 'response': parsed,
            }
            append_jsonl(attempt_path(split), {**record, 'attempt': attempt, 'raw_response': response.output_text})
            return record
        except Exception as exc:
            append_jsonl(attempt_path(split), {
                'instance_id': str(row['id']), 'split': split, 'attempt': attempt,
                'created_utc': timestamp, 'error_type': type(exc).__name__, 'error': str(exc),
            })
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

def validate_cache_record(record: dict) -> dict:
    expected = {
        'prompt_version': PROMPT_VERSION, 'prompt_hash': short_hash(CI_SYSTEM_PROMPT),
        'schema_version': SCHEMA_VERSION, 'model': CI_MODEL,
        'reasoning_effort': CI_REASONING_EFFORT, 'image_detail': CI_IMAGE_DETAIL,
        'demonstrations': [],
    }
    for key, value in expected.items():
        if record.get(key) != value:
            raise ValueError(f'Incompatible cache field {key!r} for {record.get("instance_id")}')
    copy = dict(record)
    copy['response'] = validate_response(record['response'])
    return copy

def load_or_infer(frame: pd.DataFrame, split: str, run_calls: bool) -> pd.DataFrame:
    path = cache_path(split)
    cached = {}
    for record in load_jsonl(path):
        record = validate_cache_record(record)
        key = str(record['instance_id'])
        if key in cached:
            raise ValueError(f'Duplicate valid cache ID {key!r}')
        cached[key] = record

    rows = frame.head(RUN_LIMIT) if RUN_LIMIT is not None else frame
    print(f'{split} valid cache: {sum(str(x) in cached for x in rows.id)}/{len(rows)}')
    if run_calls:
        v1_by_group = v1_caption_map(frame)
        for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f'CI {split}'):
            key = str(row['id'])
            if key not in cached:
                record = call_ci_judge(row, v1_by_group[str(row['base_id'])], split)
                append_jsonl(path, record)
                cached[key] = record
                time.sleep(REQUEST_SLEEP_SECONDS)
    ordered = [cached[str(x)] for x in rows['id'] if str(x) in cached]
    return pd.DataFrame(ordered)


In [ ]:
def safe_spearman(y, p) -> float:
    value = spearmanr(np.asarray(y, float), np.asarray(p, float)).statistic
    return float(value) if np.isfinite(value) else -1.0

def metrics(y, p) -> dict:
    return {'spearman': safe_spearman(y, p), 'mae': float(mean_absolute_error(y, p))}

def version_prior_oof(frame: pd.DataFrame, n_splits=6, statistic='median') -> np.ndarray:
    output = np.zeros(len(frame), float)
    groups = frame['base_id'].to_numpy()
    for tr, va in GroupKFold(n_splits).split(frame, groups=groups):
        part = frame.iloc[tr]
        table = part.groupby('caption_version')['CRAI_CI'].agg(statistic).to_dict()
        output[va] = frame.iloc[va]['caption_version'].map(table).to_numpy(float)
    return output

preflight_prior = version_prior_oof(train_df, n_splits=6, statistic='median')
preflight = metrics(train_df['CRAI_CI'], preflight_prior)
preflight_pass = (
    preflight['spearman'] >= PRE_API_MIN_VERSION_PRIOR_RHO and
    preflight['mae'] <= PRE_API_MAX_VERSION_PRIOR_MAE
)

print('Zero-API grouped version-median prior:', preflight)
print('PRE-API PREFLIGHT:', 'PASS' if preflight_pass else 'STOP')
assert abs(preflight['spearman'] - 0.7094332264682706) < 1e-9
assert abs(preflight['mae'] - 0.1774166666666667) < 1e-9
assert preflight_pass


In [ ]:
train_records = load_or_infer(train_df, 'train', RUN_TRAIN_API_CALLS)

                                                                              
                                              
dev_records = load_or_infer(dev_df, 'dev', False)
test_records = load_or_infer(test_df, 'test', False) if len(test_df) else pd.DataFrame()


In [ ]:
NUMERIC_FEATURES = REQUIRED_NUMERIC.copy()

def records_to_features(records: pd.DataFrame, metadata: pd.DataFrame) -> pd.DataFrame:
    if len(records) == 0:
        return pd.DataFrame()
    output = []
    for _, record in records.iterrows():
        response = validate_response(record['response'])
        row = {'id': str(record['instance_id']), **{x: response[x] for x in REQUIRED_NUMERIC}}
        row['integrity_failure'] = response['integrity_failure']
        row['brief_reason'] = response['brief_reason']
        output.append(row)
    features = metadata[['id', 'base_id', 'caption_version', 'caption_version_key']].merge(
        pd.DataFrame(output), on='id', how='inner', validate='one_to_one'
    )
    assert not features[NUMERIC_FEATURES].isna().any().any()
    return features

train_features = records_to_features(train_records, train_df)
dev_features = records_to_features(dev_records, dev_df)
test_features = records_to_features(test_records, test_df) if len(test_df) else pd.DataFrame()

print('Feature coverage:', len(train_features), len(dev_features), len(test_features))
if len(train_features):
    display(train_features.head())


In [ ]:
FAMILIES = [
    'raw_ci', 'version_prior', 'score_blend', 'rank_blend',
    'ridge_visual', 'ridge_full',
]

def add_gold(features: pd.DataFrame, gold: pd.DataFrame) -> pd.DataFrame:
    return features.merge(
        gold[['id', 'CRAI_CI']], on='id', how='inner', validate='one_to_one'
    ).rename(columns={'CRAI_CI': 'gold_ci'})

def fixed_design(frame: pd.DataFrame, include_version: bool) -> np.ndarray:
    columns = [frame[x].to_numpy(float) for x in NUMERIC_FEATURES]
    for level in FAILURE_LEVELS:
        columns.append(frame['integrity_failure'].eq(level).to_numpy(float))
    if include_version:
        for version in range(1, 6):
            columns.append(frame['caption_version'].eq(version).to_numpy(float))
    return np.column_stack(columns)

def fit_ridge_predict(train, valid, alpha, include_version):
    Xtr = fixed_design(train, include_version)
    Xva = fixed_design(valid, include_version)
    scaler = StandardScaler().fit(Xtr)
    model = Ridge(alpha=float(alpha)).fit(scaler.transform(Xtr), train['gold_ci'].to_numpy(float))
    return np.clip(model.predict(scaler.transform(Xva)), 0.0, 1.0), (scaler, model)

def empirical_percentile(reference, values):
    reference = np.sort(np.asarray(reference, float))
    return np.searchsorted(reference, np.asarray(values, float), side='right') / max(len(reference), 1)

def fit_prior(train, valid):
    table = train.groupby('caption_version')['gold_ci'].median().to_dict()
    return valid['caption_version'].map(table).to_numpy(float), table

def predict_prior(table, valid):
    return valid['caption_version'].map(table).to_numpy(float)

def choose_by_rho_then_mae(y, candidates: dict):
    rows = []
    for name, prediction in candidates.items():
        m = metrics(y, prediction)
        rows.append((name, m['spearman'], m['mae']))
    rows.sort(key=lambda x: (-x[1], x[2], str(x[0])))
    return rows[0][0], pd.DataFrame(rows, columns=['parameter', 'spearman', 'mae'])

def inner_oof_and_outer_prediction(outer_train, outer_valid, family):
    groups = outer_train['base_id'].to_numpy()
    y = outer_train['gold_ci'].to_numpy(float)
    splits = list(GroupKFold(min(INNER_FOLDS, outer_train['base_id'].nunique())).split(outer_train, groups=groups))

    raw_oof = outer_train['raw_ci'].to_numpy(float)
    raw_valid = outer_valid['raw_ci'].to_numpy(float)

    prior_oof = np.zeros(len(outer_train), float)
    for tr, va in splits:
        pred, _ = fit_prior(outer_train.iloc[tr], outer_train.iloc[va])
        prior_oof[va] = pred
    prior_valid, prior_table = fit_prior(outer_train, outer_valid)

    if family == 'raw_ci':
        return raw_oof, raw_valid, {'family': family}
    if family == 'version_prior':
        return prior_oof, prior_valid, {'family': family, 'prior': prior_table}

    if family in {'score_blend', 'rank_blend'}:
        candidate_oof = {}
        valid_by_weight = {}
        for weight in BLEND_WEIGHTS:
            if family == 'score_blend':
                candidate_oof[weight] = weight * raw_oof + (1.0 - weight) * prior_oof
                valid_by_weight[weight] = weight * raw_valid + (1.0 - weight) * prior_valid
            else:
                rank_oof = np.zeros(len(outer_train), float)
                for tr, va in splits:
                    train_part, valid_part = outer_train.iloc[tr], outer_train.iloc[va]
                    p_va, p_table = fit_prior(train_part, valid_part)
                    p_tr = predict_prior(p_table, train_part)
                    rr = empirical_percentile(train_part['raw_ci'], valid_part['raw_ci'])
                    rp = empirical_percentile(p_tr, p_va)
                    rank_oof[va] = weight * rr + (1.0 - weight) * rp
                raw_rank_valid = empirical_percentile(outer_train['raw_ci'], outer_valid['raw_ci'])
                prior_train = predict_prior(prior_table, outer_train)
                prior_rank_valid = empirical_percentile(prior_train, prior_valid)
                candidate_oof[weight] = rank_oof
                valid_by_weight[weight] = weight * raw_rank_valid + (1.0 - weight) * prior_rank_valid
        best_weight, _ = choose_by_rho_then_mae(y, candidate_oof)
        return candidate_oof[best_weight], valid_by_weight[best_weight], {
            'family': family, 'weight': float(best_weight), 'prior': prior_table,
        }

    include_version = family == 'ridge_full'
    candidate_oof = {}
    for alpha in RIDGE_ALPHAS:
        oof = np.zeros(len(outer_train), float)
        for tr, va in splits:
            oof[va], _ = fit_ridge_predict(
                outer_train.iloc[tr], outer_train.iloc[va], alpha, include_version
            )
        candidate_oof[alpha] = oof
    best_alpha, _ = choose_by_rho_then_mae(y, candidate_oof)
    valid_prediction, fitted = fit_ridge_predict(
        outer_train, outer_valid, best_alpha, include_version
    )
    return candidate_oof[best_alpha], valid_prediction, {
        'family': family, 'alpha': float(best_alpha), 'fitted': fitted,
    }

def fit_calibrator(name, prediction, y):
    prediction = np.asarray(prediction, float)
    y = np.asarray(y, float)
    if name == 'identity':
        return {'name': name}
    if name == 'positive_affine':
        model = LinearRegression(positive=True).fit(prediction.reshape(-1, 1), y)
        return {'name': name, 'model': model}
    if name == 'monotone_logit':
        z = logit(np.clip(prediction, 1e-4, 1 - 1e-4))
        best = None
        for shift in np.arange(-2.0, 2.01, 0.25):
            for slope in np.arange(0.5, 2.51, 0.25):
                p = expit(shift + slope * z)
                score = mean_absolute_error(y, p)
                if best is None or score < best[0]:
                    best = (score, float(shift), float(slope))
        return {'name': name, 'shift': best[1], 'slope': best[2]}
    if name == 'isotonic':
        model = IsotonicRegression(out_of_bounds='clip').fit(prediction, y)
        return {'name': name, 'model': model}
    raise ValueError(name)

def apply_calibrator(calibrator, prediction):
    prediction = np.asarray(prediction, float)
    name = calibrator['name']
    if name == 'identity':
        output = prediction
    elif name == 'positive_affine':
        output = calibrator['model'].predict(prediction.reshape(-1, 1))
    elif name == 'monotone_logit':
        z = logit(np.clip(prediction, 1e-4, 1 - 1e-4))
        output = expit(calibrator['shift'] + calibrator['slope'] * z)
    elif name == 'isotonic':
        output = calibrator['model'].predict(prediction)
    else:
        raise ValueError(name)
    return np.clip(output, 0.0, 1.0)

def select_calibrator(base_prediction, y):
    base_rho = safe_spearman(y, base_prediction)
    rows, fitted = [], {}
    for name in ['identity', 'positive_affine', 'monotone_logit', 'isotonic']:
        calibrator = fit_calibrator(name, base_prediction, y)
        prediction = apply_calibrator(calibrator, base_prediction)
        m = metrics(y, prediction)
        eligible = m['spearman'] >= base_rho - 0.01
        rows.append({'calibrator': name, **m, 'eligible': eligible})
        fitted[name] = calibrator
    report = pd.DataFrame(rows)
    eligible = report[report['eligible']].sort_values(['mae', 'spearman'], ascending=[True, False])
    best_name = str(eligible.iloc[0]['calibrator'])
    return fitted[best_name], report

def nested_grouped_search(training):
    groups = training['base_id'].to_numpy()
    y = training['gold_ci'].to_numpy(float)
    outer = list(GroupKFold(OUTER_FOLDS).split(training, groups=groups))
    oof = {family: np.zeros(len(training), float) for family in FAMILIES}
    fold_rows, parameter_rows = [], []

    for fold, (tr, va) in enumerate(outer, start=1):
        outer_train, outer_valid = training.iloc[tr].copy(), training.iloc[va].copy()
        for family in FAMILIES:
            inner_oof, outer_base, parameters = inner_oof_and_outer_prediction(
                outer_train, outer_valid, family
            )
            calibrator, calibration_report = select_calibrator(
                inner_oof, outer_train['gold_ci'].to_numpy(float)
            )
            outer_prediction = apply_calibrator(calibrator, outer_base)
            oof[family][va] = outer_prediction
            m = metrics(outer_valid['gold_ci'], outer_prediction)
            fold_rows.append({'fold': fold, 'family': family, **m,
                              'calibrator': calibrator['name']})
            parameter_rows.append({
                'fold': fold, 'family': family,
                'rank_parameters': json.dumps({k: v for k, v in parameters.items() if k != 'fitted'}, default=float),
                'calibrator': calibrator['name'],
            })

    summary = []
    for family, prediction in oof.items():
        m = metrics(y, prediction)
        fold_part = pd.DataFrame(fold_rows).query('family == @family')
        summary.append({'family': family, **m,
                        'fold_rho_mean': fold_part['spearman'].mean(),
                        'fold_rho_std': fold_part['spearman'].std(ddof=1)})
    return {
        'oof': oof, 'summary': pd.DataFrame(summary).sort_values(
            ['spearman', 'mae'], ascending=[False, True]
        ).reset_index(drop=True),
        'folds': pd.DataFrame(fold_rows),
        'parameters': pd.DataFrame(parameter_rows),
    }


In [ ]:
search_result = None
train_gate = None
selected_family = None

if len(train_features) == len(train_df):
    training = add_gold(train_features, train_df)
    search_result = nested_grouped_search(training)
    display(search_result['summary'].round(4))

    selected_family = str(search_result['summary'].iloc[0]['family'])
    selected_row = search_result['summary'].iloc[0]
    prior_row = search_result['summary'].query("family == 'version_prior'").iloc[0]
    folds = search_result['folds']
    selected_folds = folds.query('family == @selected_family').set_index('fold')
    prior_folds = folds.query("family == 'version_prior'").set_index('fold')
    fold_wins = int((selected_folds['spearman'] > prior_folds['spearman']).sum())
    delta_rho = float(selected_row['spearman'] - prior_row['spearman'])

    train_gate = (
        float(selected_row['spearman']) >= TRAIN_GATE_MIN_RHO and
        delta_rho >= TRAIN_GATE_MIN_DELTA_RHO and
        float(selected_row['mae']) <= TRAIN_GATE_MAX_MAE and
        fold_wins >= TRAIN_GATE_MIN_FOLD_WINS
    )
    gate_table = pd.DataFrame([{
        'selected_family': selected_family,
        'oof_spearman': float(selected_row['spearman']),
        'oof_mae': float(selected_row['mae']),
        'version_prior_spearman': float(prior_row['spearman']),
        'delta_spearman': delta_rho,
        'fold_wins_out_of_5': fold_wins,
        'decision': 'PASS — dev calls justified' if train_gate else 'STOP — do not run dev calls',
    }])
    display(gate_table.round(4))
    search_result['summary'].to_csv(OUTPUT_DIR / 'ci_train_nested_cv_summary.tsv', sep='\t', index=False)
    search_result['folds'].to_csv(OUTPUT_DIR / 'ci_train_nested_cv_folds.tsv', sep='\t', index=False)
    search_result['parameters'].to_csv(OUTPUT_DIR / 'ci_train_selected_parameters.tsv', sep='\t', index=False)
else:
    print(f'Train cache incomplete: {len(train_features)}/{len(train_df)}. Run train API calls first.')


In [ ]:
final_fit = None

def fit_selected_on_all(training, target, family):
    inner_oof, target_base, parameters = inner_oof_and_outer_prediction(training, target, family)
    calibrator, calibration_report = select_calibrator(
        inner_oof, training['gold_ci'].to_numpy(float)
    )
    prediction = apply_calibrator(calibrator, target_base)
    return {
        'prediction': prediction, 'parameters': parameters,
        'calibrator': calibrator, 'calibration_report': calibration_report,
        'train_inner_oof': inner_oof,
    }

if RUN_DEV_API_CALLS:
    if train_gate is not True:
        print('REFUSING DEV API CALLS: the predeclared train-only gate did not pass.')
    else:
        dev_records = load_or_infer(dev_df, 'dev', True)
        dev_features = records_to_features(dev_records, dev_df)

if len(dev_features) == len(dev_df) and selected_family is not None and len(train_features) == len(train_df):
    if train_gate is not True:
        print('Dev cache exists, but the predeclared train gate did not pass. Results are diagnostic only.')
    training = add_gold(train_features, train_df)
    final_fit = fit_selected_on_all(training, dev_features, selected_family)
    dev_eval = dev_df[['id', 'base_id', 'caption_version_key', 'CRAI_CI']].copy()
    dev_eval['prediction'] = final_fit['prediction']
    dev_metrics = metrics(dev_eval['CRAI_CI'], dev_eval['prediction'])

                                                            
    prior_table = train_df.groupby('caption_version')['CRAI_CI'].median().to_dict()
    dev_eval['version_median_prior'] = dev_df['caption_version'].map(prior_table).to_numpy(float)
    prior_dev_metrics = metrics(dev_eval['CRAI_CI'], dev_eval['version_median_prior'])

    comparison = pd.DataFrame([
        {'system': 'Historical two-stage CI', 'spearman': 0.6236218974, 'mae': 0.2749166667},
        {'system': 'Train-fitted version-median prior', **prior_dev_metrics},
        {'system': f'New structured + {selected_family}', **dev_metrics},
    ])
    display(comparison.round(4))
    display(final_fit['calibration_report'].round(4))
    dev_eval.to_csv(OUTPUT_DIR / 'ci_dev_predictions.tsv', sep='\t', index=False)
    comparison.to_csv(OUTPUT_DIR / 'ci_dev_comparison.tsv', sep='\t', index=False)
else:
    print(f'Dev cache incomplete: {len(dev_features)}/{len(dev_df)}.')
    if train_gate is True:
        print('Train gate passed. Set RUN_DEV_API_CALLS=True for one dev run.')
    else:
        print('Do not run dev calls until the train gate passes.')


In [ ]:
if RUN_TEST_API_CALLS:
    if train_gate is not True:
        raise RuntimeError('Refusing test API calls because the train gate did not pass.')
    test_records = load_or_infer(test_df, 'test', True)
    test_features = records_to_features(test_records, test_df)

if len(test_features) == len(test_df) and len(test_df):
    if train_gate is not True:
        raise RuntimeError('Refusing test export because the train gate did not pass.')
    training = add_gold(train_features, train_df)
    test_fit = fit_selected_on_all(training, test_features, selected_family)
    test_output = pd.DataFrame({
        'id': test_df['id'].astype(str),
        'CRAI_CI': np.clip(test_fit['prediction'], 0.0, 1.0),
    })
    test_output.to_csv(OUTPUT_DIR / 'ci_test_predictions.tsv', sep='\t', index=False)
    display(test_output.head())
else:
    print('No complete test cache; no test file written.')
